# AI Suite Python SDK demo

Ready-to-use notebook for driving the Arango AI Suite from Python with `arango-ai-sdk`. You should not have to copy gateway URLs, auth headers, or JSON bodies. The only file you edit is `env`: credentials and API keys.

This is **part 1** — every SDK step up to and including AutoGraph install:

1. Sign in
2. Create / get a project
3. Store secret profiles
4. Upload documents (`upload_files` and `upload_folder`)
5. Deploy AutoGraph

**Not in the package yet (later parts):** corpus build, strategizer, orchestrate, overview, chat, wait-until-healthy, and deleting projects / secrets / files. Those still go through the AutoGraph HTTP API or the UI.

## How to run it

1. Copy `env.example` to `env` next to this notebook and fill in gateway, ArangoDB login, a unique `PROJECT_NAME`, API keys, and `FPS_RECOVERY_USERNAME`. Do not change the notebook cells.
2. Select a Python 3.12+ kernel, then run the **preflight** cell.
3. If preflight says the SDK is missing, run the two install cells and **restart the kernel**. Otherwise skip them.
4. Run top to bottom. `./corpus` ships two small files (categories `arangodb` and `tech-companies`).

> **Use a fresh `PROJECT_NAME` for each run.** The SDK cannot delete a project yet, so re-using a name fails at step 2.

> `env` holds real credentials. Keep it out of git and off shared screens.


---
## Preflight

Check the kernel before installing anything. The SDK needs Python 3.12 or newer; some notebook images still ship 3.11.


In [ ]:
import importlib.util
import sys
from pathlib import Path

HERE = next(
    (
        p.resolve()
        for p in (
            Path.cwd() / "examples" / "sdk-demo",
            Path.cwd() / "arango-ai-sdk" / "examples" / "sdk-demo",
            Path.cwd(),
        )
        if p.name == "sdk-demo" and (p / "AI_Suite_SDK_demo.ipynb").is_file()
    ),
    None,
)
if HERE is None:
    raise SystemExit("Open this notebook from examples/sdk-demo (or the SDK repo root).")
wheels = sorted(HERE.glob("arango_ai_sdk-*.whl")) + sorted(
    (HERE / "wheel").glob("arango_ai_sdk-*.whl")
)

print("python   :", sys.version)
print("3.12+    :", sys.version_info >= (3, 12))
print("sdk here :", importlib.util.find_spec("arango_ai_sdk") is not None)
print("demo     :", HERE)
print("wheels   :", [p.name for p in wheels])

## Install *(skip when preflight says `sdk here: True` — after a wheel upgrade, re-run these two cells and restart)*

The package is on private PyPI. A cell cannot log into Google Artifact Registry, so `pip install arango-ai-sdk` from that index hangs. For this notebook: `make build` or `uv build` in the SDK repo, copy `dist/arango_ai_sdk-*.whl` next to the notebook (or into `./wheel`), run both cells, then **restart the kernel**. The wheel is a workaround; with GAR login, install from the index as in the package README.

`--no-input` fails fast if pip asks for credentials.


In [ ]:
if not wheels:
    raise SystemExit("copy arango_ai_sdk-*.whl next to this notebook (or into ./wheel)")
WHEEL = str(wheels[-1])
print("installing", WHEEL)

In [ ]:
%pip install --no-input python-dotenv
%pip install --no-input --force-reinstall "$WHEEL"

---
## 0. Configuration

`env` holds connection info and keys only. Never print keys.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

import arango_ai_sdk

HERE = next(
    (
        p.resolve()
        for p in (
            Path.cwd() / "examples" / "sdk-demo",
            Path.cwd() / "arango-ai-sdk" / "examples" / "sdk-demo",
            Path.cwd(),
        )
        if p.name == "sdk-demo" and (p / "AI_Suite_SDK_demo.ipynb").is_file()
    ),
    None,
)
if HERE is None:
    raise SystemExit("Open this notebook from examples/sdk-demo (or the SDK repo root).")
env_path = HERE / "env"
if not env_path.is_file():
    raise SystemExit("Copy env.example to env in examples/sdk-demo (next to this notebook).")
load_dotenv(env_path, override=True)

GATEWAY = (os.getenv("GATEWAY") or os.getenv("ARANGO_DEPLOYMENT_ENDPOINT") or "").rstrip("/")
USERNAME = (
    os.getenv("ARANGO_USERNAME") or os.getenv("ARANGO_SDK_TEST_USERNAME") or os.environ["USERNAME"]
)
PASSWORD = os.getenv("ARANGO_PASSWORD") or os.getenv("ARANGO_SDK_TEST_PASSWORD")
DB = os.getenv("DB_NAME") or os.getenv("ARANGO_SDK_TEST_DB")
if not GATEWAY or not PASSWORD or not DB:
    raise SystemExit("Set GATEWAY, ARANGO_PASSWORD and DB_NAME in env.")
PROJECT = os.environ["PROJECT_NAME"]
CHAT_API_KEY = os.environ["CHAT_API_KEY"]
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY") or CHAT_API_KEY
CHAT_PROVIDER = os.getenv("CHAT_API_PROVIDER", "openai")
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_API_PROVIDER", "openai")
CHAT_MODEL = os.getenv("CHAT_MODEL") or None
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL") or None
FPS_RECOVERY_USERNAME = os.getenv("FPS_RECOVERY_USERNAME") or USERNAME
CORPUS_DIR = (HERE / os.getenv("CORPUS_DIR", "./corpus")).resolve()

print("sdk     :", arango_ai_sdk.__version__)
print("gateway :", GATEWAY)
print("db      :", DB)
print("project :", PROJECT)
print("corpus  :", CORPUS_DIR)

---
## The same flow without the SDK

Until now, driving the suite from Python meant talking to the gateway yourself over HTTP with a JWT. The snippets below are what that costs you. They are here for reference only — **nothing in this section runs**.

You start as a client of the gateway rather than a user of a library, so you own the plumbing:

```python
# Reference only — not executed
S = requests.Session()

class ApiError(RuntimeError):
    def __init__(self, method, path, status_code, body):
        super().__init__(f"{method} {path} -> {status_code}: {body[:600]}")
        self.status_code = status_code

def authenticate():
    r = S.post(f"{GATEWAY}/_db/_system/_open/auth",
               json={"username": USERNAME, "password": PASSWORD})
    r.raise_for_status()
    S.headers.update({"Authorization": f"Bearer {r.json()['jwt']}"})

def api(method, path, *, base=None, **kw):
    url = (base or GATEWAY) + path
    r = S.request(method, url, **kw)
    if not r.ok:
        raise ApiError(method, path, r.status_code, r.text)
    return r.json() if r.content else {}
```

**The JWT lasts about 60 minutes,** so you re-authenticate before anything slow. Forget it and a long upload dies with a 401 halfway through.

**You have to know the internal layout.** A wrong prefix is a silent 404:

- `POST /_db/_system/_open/auth`
- `POST /_platform/acp/v1/project`
- `GET /_platform/acp/v1/project_by_name/{db}/{project}`
- `POST /_platform/acp/v1/secrets`
- File Manager `POST .../rag-input/batch` (multipart)
- AutoGraph `{AG}/v1/corpus/builds`, `/rag-strategizer/analyze`, `/orchestrate`

**The wire format leaks into your code.** Responses mix camelCase and snake_case (`category` vs the older `module`), so you write a reader for both:

```python
# Reference only — not executed
def _camel(name: str):
    return re.sub(r"_(\w)", lambda m: m.group(1).upper(), name)

def pick(obj, *names, default=None):
    for name in names:
        for key in (name, _camel(name)):
            if isinstance(obj, dict) and key in obj:
                return obj[key]
    return default
```

The upload scope has to be `[project, category]`, or the corpus build later cannot resolve the files.

**Every long-running job is a polling loop you write yourself** — deploy, corpus build, strategizer, orchestration, each with its own timeouts and a 404 while the service is still coming up:

```python
# Reference only — not executed
def wait_until(poll_fn, is_done, desc="", every=10, timeout=7200):
    ...
```

**Uploading is a multipart body, and deploying is an install `env` map** — not "upload a folder" and "start AutoGraph":

```python
# Reference only — not executed
def upload_batch(files_by_category: dict[str, list[Path]]):
    # files parts + scope fields, path.read_bytes()
    ...

api("POST", "/_platform/acp/v1/service/...", json={"env": {
    "GENAI_PROJECT_NAME": PROJECT,
    "CHAT_SECRET_PROFILE_ID": chat_id,
    "FPS_RECOVERY_USERNAME": FPS_USER,
    # providers, models, ...
}})
```

**Errors are a status code and a body.** You catch one generic error and branch on `e.status_code` to work out what actually went wrong.

The rest of this notebook does the same five steps as method calls.


---
## The same flow with the SDK

| Step | Over HTTP | With the SDK |
| --- | --- | --- |
| Sign in | `authenticate()` + Bearer header on every call | `ArangoAIClient(url, user, password, db_name)` |
| Project | `POST /_platform/acp/v1/project`, then read the JSON | `client.project.create` returns a project handle |
| Secrets | `POST /_platform/acp/v1/secrets` | `client.secret_profile.create` / `.get` |
| Upload | Hand-built multipart (`/rag-input/batch`) | `client.files.upload_files(...)` or `client.files.upload_folder(...)` |
| Deploy | Install `env` map | `project.autograph.deploy` / `.undeploy` |
| Model config | `PUT {AG}/v1/projects/{project}/model-config/credentials` | not in the package yet |
| Corpus build | `POST {AG}/v1/corpus/builds` + poll | `project.autograph.build_corpus(...)` — not in the package yet |
| Strategizer | `POST {AG}/v1/rag-strategizer/analyze` + poll | `project.autograph.strategize(...)` — not in the package yet |
| Orchestrate | `POST {AG}/v1/orchestrate` + poll | `project.autograph.orchestrate(...)` — not in the package yet |
| Wait | `wait_until(...)` copied per job | `project.autograph.wait(...)` — not in the package yet |
| Chat | Retriever HTTP (`LOCAL` / `GLOBAL` / `UNIFIED`) | `project.retriever.chat(...)` — not in the package yet |

The first five rows run in this notebook. The rest is the rest of the pipeline, so you can see what still lives on HTTP.

Install the package, pass your credentials once, call named methods.

---
## 1. Sign in

Constructing the client authenticates. The token and its renewal live inside the SDK, so nothing below sets a header. Bad credentials fail here with `InvalidCredentialsError`.


In [ ]:
from arango_ai_sdk import ArangoAIClient

client = ArangoAIClient(
    base_url=GATEWAY,
    username=USERNAME,
    password=PASSWORD,
    db_name=DB,
)
print("signed in as:", client.username)
print("database    :", client.db_name)

---
## 2. Create the project

`create` returns a project handle, an object with `.name`, `.file_names`, and services such as `.autograph` — instead of a JSON dict you have to read key by key. `get` fetches an existing one.

Use a fresh `PROJECT_NAME` in `env` so re-runs do not collide.


In [ ]:
project = client.project.create(
    project_name=PROJECT,
    project_type="autograph",
    description="Created from the AI Suite SDK demo notebook.",
)
fetched = client.project.get(project_name=PROJECT)
print("created:", project.name, project.project_type, project.db_name)
print("fetched:", fetched.name, fetched.project_type)

---
## 3. Store LLM credentials as secret profiles

The raw API keys go to the platform once. Every later step refers to them by `profile_id`, so the key never travels through your code again. Print the ids, never the keys.


In [ ]:
chat = client.secret_profile.create(
    name=f"{PROJECT}-chat",
    secret=CHAT_API_KEY,
    provider=CHAT_PROVIDER,
)
embed = client.secret_profile.create(
    name=f"{PROJECT}-embed",
    secret=EMBEDDING_API_KEY,
    provider=EMBEDDING_PROVIDER,
)
print("chat :", chat.profile_id)
print("embed:", embed.profile_id)
print("get  :", client.secret_profile.get(chat.profile_id).name)

---
## 4. Upload files

Documents live under `scope = [project, category]`. The SDK has two upload APIs. This cell runs **both**, each on a different category, so you see a real result from each without uploading the same file twice.

**`upload_folder`** — files sitting *directly* in one folder, all under one scope. Subfolders are ignored. Used here for `corpus/arangodb/`.

**`upload_files`** — several files in one request: a list that shares one `scope`, or a mapping of path → scope when categories differ. Used here for `corpus/tech-companies/` (list + shared scope). One request is at most 100 files or 2 GiB.

Do not call `upload_folder(CORPUS_DIR)` on the corpus root — that would skip both files because they live in subfolders. There is also `upload_file` for a single path.

Each result should come back `status == "ok"`. Listing, downloading, and deleting files are not in the SDK yet.


In [ ]:
# upload_folder: one flat category directory
arangodb_dir = CORPUS_DIR / "arangodb"
folder_results = client.files.upload_folder(arangodb_dir, scope=[PROJECT, arangodb_dir.name])
print("upload_folder", arangodb_dir.name)
for r in folder_results:
    print(" ", r.name, r.status, r.scope or r.detail)

# upload_files: remaining category as a list that shares one scope
tech_dir = CORPUS_DIR / "tech-companies"
tech_files = sorted(p for p in tech_dir.iterdir() if p.is_file() and not p.name.startswith("."))
if not tech_files:
    raise SystemExit(f"no files in {tech_dir}")
file_results = client.files.upload_files(tech_files, scope=[PROJECT, tech_dir.name])
print("upload_files", tech_dir.name)
for r in file_results:
    print(" ", r.name, r.status, r.scope or r.detail)

---
## 5. Deploy AutoGraph

The same step as configuring and installing the service in the UI, with the install parameters as keyword arguments.

Success here means the platform accepted the install: `service_id` is set and `is_deployed` is `True`. The SDK does not wait for the service to become healthy, so the pod may still be starting. `service_url` is an in-cluster address, not something you open in a browser. The project also stays on the Documents step in the UI, because advancing it needs the corpus build.

Re-running this on a project that already has AutoGraph raises `AutoGraphAlreadyDeployedError`.

`fps_recovery_username` must already exist with read/write on `DB_NAME`. The SDK does not create that user.


In [ ]:
info = project.autograph.deploy(
    chat_api_provider=CHAT_PROVIDER,
    embedding_api_provider=EMBEDDING_PROVIDER,
    chat_secret_profile_id=chat.profile_id,
    embedding_secret_profile_id=embed.profile_id,
    fps_recovery_username=FPS_RECOVERY_USERNAME,
    chat_model=CHAT_MODEL,
    embedding_model=EMBEDDING_MODEL,
)
print("service_id :", info.service_id)
print("is_deployed:", project.autograph.is_deployed)
print("service_url:", project.autograph.service_url)

---
## Failures have names

Over HTTP you catch one error and branch on its status code. The SDK raises specific exceptions — `InvalidCredentialsError`, `InvalidProjectNameError`, `AutoGraphAlreadyDeployedError` all under `ArangoAIError` if you want to catch everything at once.


The next cell signs in with a wrong password. It creates and changes nothing on the platform.


In [ ]:
from arango_ai_sdk.exceptions import InvalidCredentialsError

try:
    ArangoAIClient(GATEWAY, USERNAME, "wrong-password", db_name=DB)
except InvalidCredentialsError as e:
    print(type(e).__name__, "— caught by name, no status code to inspect")

---
## What comes next

Ingestion is a configuration step followed by three jobs — corpus build, strategizer, orchestration — each of which is a POST plus a polling GET over HTTP today. The SDK will expose them as methods with a single wait, rather than three loops:

```python
# Not implemented yet — these calls do not exist
# project.autograph.configure_models(...)
# project.autograph.build_corpus(...)
# project.autograph.strategize(...)
# project.autograph.orchestrate(...)
# project.autograph.wait(...)
# project.retriever.chat(...)
```

That is the same order as the UI: Documents, Configure, Build, then chat. Until those land, use the AutoGraph HTTP API from the corpus build onward, or the UI.

Also still missing: waiting for the service to report healthy, the project overview, listing and deleting files, and deleting projects, secrets, and categories.



---
## Clean up

Remove the AutoGraph service, then close the client. The SDK cannot delete projects, secret profiles, or files yet.


In [ ]:
removed = project.autograph.undeploy()
print("undeployed:", removed.service_id, "is_deployed:", project.autograph.is_deployed)
client.close()
print("client closed")